In [2]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset --------
class ModeAndFeatureDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None, max_items=None):
        self.items = []
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if not sid.isdigit():
                continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) == 0: continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann['sequence']
            min_len = min(len(files), len(seq))
            if min_len < 15: continue

            own_speeds = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt_speeds = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            angles = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            dists_all = [self.distances.get(sid, {}).get(str(i), 0.0) for i in range(min_len)]
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                if max_items and len(self.items) >= max_items:
                    return

                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]
                t = tgt_speeds[i:i+15]

                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                rel_acc = d2 - np.mean(d2)

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                rel_speed_seq = t - s
                rel_speed_feat = rel_speed_seq[:14]
                rel_speed_feat = np.pad(rel_speed_feat, (0, 1), mode='constant')
                rel_speed = np.mean(rel_speed_seq)

                feature = np.concatenate([
                    modes, d, s, a, s1, d1, d2, rel_acc, rel_speed_feat, *d_smooths
                ])
                self.items.append((feature.astype(np.float32), rel_speed, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feature, tgt, sid = self.items[idx]
        return torch.tensor(feature), torch.tensor(tgt, dtype=torch.float32), sid

# -------- Collate --------
def collate_fn(batch):
    feats, tgts, sids = zip(*batch)
    return torch.stack(feats), torch.tensor(tgts), list(sids)

# -------- モデル --------
class ExtendedLSTMWithAttention(nn.Module):
    def __init__(self, input_dim=15, feature_dim=15, hidden_size=128):
        super().__init__()
        self.pre_fc = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=hidden_size,
                            num_layers=2, batch_first=True, dropout=0.3)

        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.dropout = nn.Dropout(0.3)
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, F = x.shape
        x = x.view(B, 15, -1)
        x = self.pre_fc(x.reshape(-1, x.size(2))).view(B, 15, -1)
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attn_fc(lstm_out), dim=1)
        context = (attn_weights * lstm_out).sum(dim=1)
        return self.fc_out(context).squeeze(1)

# -------- 学習ループ --------
def train_lstm_model(dataset, save_path="model_lstm_attn.pth"):
    scenes = sorted(list(set([item[-1] for item in dataset.items])))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx[:6000])
    val_ds = Subset(dataset, val_idx[:1500])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_dim = train_ds[0][0].shape[0] // 15
    model = ExtendedLSTMWithAttention(input_dim=15, feature_dim=input_dim).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 100
    patience_counter = 0

    for epoch in range(200):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- 実行 --------
if __name__ == "__main__":
    crop_root = "../train/train_crops"
    annot_root = "../train/train_annotations"
    distance_json_path ="../train_retry/trainestimates1_smoothed"

    dataset = ModeAndFeatureDataset(
        crop_root=crop_root,
        annot_root=annot_root,
        distance_json_path=distance_json_path,
        max_items=7500
    )

    model = train_lstm_model(dataset, save_path="model_lstm_1.pth")


IsADirectoryError: [Errno 21] Is a directory: '../train_retry/trainestimates1_smoothed'

In [2]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ======== 設定 ========
CROP_ROOT = "../test/test_crops"
ANNOT_ROOT = "../test/test_annotations"
DISTANCE_JSON = "../testdistance_estimates_filtered_renamed.json"
MODEL_PATH = "../model_lstm_attn.pth"
OUTPUT_JSON = "../submission/submission_tgtspeed.json"

# ======== Dataset（構造・特徴量一致）========
class ModeAndFeatureDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None):
        self.items = []
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if not sid.isdigit():
                continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann["sequence"]
            min_len = min(len(files), len(seq))
            if min_len < 15: continue

            own_speeds = np.array([f["OwnSpeed"] for f in seq], dtype=np.float32)
            angles = np.array([f["StrDeg"] for f in seq], dtype=np.float32)
            dists_all = [self.distances.get(sid, {}).get(files[i], 0.0) for i in range(min_len)]
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]

                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                rel_acc = d2 - np.mean(d2)

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                rel_speed_feat = np.zeros(15, dtype=np.float32)  # testでは使用不可のため0埋め
                feature = np.concatenate([
                    modes, d, s, a, s1, d1, d2, rel_acc, rel_speed_feat, *d_smooths
                ])
                self.items.append((feature.astype(np.float32), sid, np.mean(s)))  # own speed も渡す

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, sid, own = self.items[idx]
        return torch.tensor(feat), sid, own

# ======== モデル（学習構造に完全一致）========
class ExtendedLSTMWithAttention(nn.Module):
    def __init__(self, input_dim=15, feature_dim=16, hidden_size=128):  # ←★重要：feature_dim=16 に修正
        super().__init__()
        self.pre_fc = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=hidden_size,
                            num_layers=2, batch_first=True, dropout=0.3)
        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Tanh(),  # ←学習コードに合わせてTanh
            nn.Linear(64, 1)
        )
        self.dropout = nn.Dropout(0.3)
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, F = x.shape
        x = x.view(B, 15, -1)
        x = self.pre_fc(x.reshape(-1, x.size(2))).view(B, 15, -1)
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attn_fc(lstm_out), dim=1)
        context = (attn_weights * lstm_out).sum(dim=1)
        return self.fc_out(context).squeeze(1)

# ======== 推論処理 ========
def inference():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataset = ModeAndFeatureDataset(CROP_ROOT, ANNOT_ROOT, DISTANCE_JSON)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=False)

    input_dim = 16  # ← ★ここも学習時の特徴次元に固定！
    model = ExtendedLSTMWithAttention(input_dim=15, feature_dim=input_dim).to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()

    output = {}
    with torch.no_grad():
        for feats, sids, owns in tqdm(dataloader, desc="🔍 Inference"):
            feats = feats.to(device)
            preds_rel = model(feats).cpu().numpy()
            owns = owns.numpy()
            preds_abs = preds_rel + owns

            for sid, pred in zip(sids, preds_abs):
                output.setdefault(sid, []).append(float(pred))

    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    print(f"✅ 推論完了: {OUTPUT_JSON} に保存しました")

# ======== 実行 ========
if __name__ == "__main__":
    inference()


RuntimeError: Error(s) in loading state_dict for ExtendedLSTMWithAttention:
	Missing key(s) in state_dict: "pre_fc.2.weight", "pre_fc.2.bias", "attn_fc.0.weight", "attn_fc.0.bias", "attn_fc.2.weight", "attn_fc.2.bias". 
	Unexpected key(s) in state_dict: "attn.0.weight", "attn.0.bias", "attn.2.weight", "attn.2.bias", "pre_fc.1.weight", "pre_fc.1.bias", "pre_fc.1.running_mean", "pre_fc.1.running_var", "pre_fc.1.num_batches_tracked", "pre_fc.3.weight", "pre_fc.3.bias". 